# 09. SacreBLEU, chrF++, & COMET Benchmark Evaluation

**Requires GPU.** Evaluates trained QLoRA model checkpoints (or zero-shot base models) against the fixed `master_test.csv` split across SacreBLEU, chrF++, Lexical Term Accuracy, and neural COMET.

In [ ]:
# ============================================================
# PATH & ENVIRONMENT BOOSTER — Guarantees project path setup
# ============================================================
import os, sys, site, glob

user_site = site.getusersitepackages()
if user_site not in sys.path:
    sys.path.insert(0, user_site)
for conda_site in glob.glob('/opt/conda/lib/python3.*/site-packages'):
    if conda_site not in sys.path:
        sys.path.insert(0, conda_site)

if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

print(f'Working Directory : {os.getcwd()}')
print(f'Python Kernel     : {sys.executable}')


In [ ]:
import glob, os

MODEL_NAME = 'qwen'                     # 'qwen' or 'mistral'
EXPERIMENT_ID = 'E7_Curriculum_Learning' # e.g. 'E1_English_Ekegusii', 'E7_Curriculum_Learning', 'E9_Sequential_Transfer', or None for E0 Baseline

ADAPTER_PATH = None
if EXPERIMENT_ID:
    ckpt_dir = f'checkpoints/{MODEL_NAME}/{EXPERIMENT_ID}'
    subdirs = sorted(glob.glob(f'{ckpt_dir}/checkpoint-*'), key=lambda x: int(x.split('-')[-1]))
    if subdirs:
        ADAPTER_PATH = subdirs[-1]
    elif os.path.exists(f'{ckpt_dir}/best'):
        ADAPTER_PATH = f'{ckpt_dir}/best'
    elif os.path.exists(ckpt_dir):
        ADAPTER_PATH = ckpt_dir

print(f'🎯 Target Model        : {MODEL_NAME}')
print(f'🧪 Experiment ID       : {EXPERIMENT_ID or "E0_Baseline (Zero-Shot)"}')
print(f'📦 Resolved Adapter Path: {ADAPTER_PATH}')


In [ ]:
from src.cli.evaluate import run_evaluate

# Evaluate English -> Ekegusii
print("📊 Evaluating English -> Ekegusii...")
results_eng_eke = run_evaluate(MODEL_NAME, source_lang='English', target_lang='Ekegusii', adapter_path=ADAPTER_PATH)
print("Results (English -> Ekegusii):", results_eng_eke)

# Evaluate Ekegusii -> English
print("\n📊 Evaluating Ekegusii -> English...")
results_eke_eng = run_evaluate(MODEL_NAME, source_lang='Ekegusii', target_lang='English', adapter_path=ADAPTER_PATH)
print("Results (Ekegusii -> English):", results_eke_eng)


## Save evaluation results for the ablation study

In [ ]:
import json
from pathlib import Path

out_dir = Path('experiments') / (EXPERIMENT_ID or 'E0_Baseline')
out_dir.mkdir(parents=True, exist_ok=True)
results_path = out_dir / 'results.json'

existing = json.loads(results_path.read_text()) if results_path.exists() else {'experiment_id': EXPERIMENT_ID or 'E0_Baseline'}
existing[MODEL_NAME] = {
    'eng_to_eke': results_eng_eke,
    'eke_to_eng': results_eke_eng
}
results_path.write_text(json.dumps(existing, indent=2))
print(f'✅ Results saved to {results_path}')


## Neural COMET Metric Evaluation (Optional)

In [ ]:
from src.experiments.base import BaseExperiment
from src.master_corpus.manager import MasterCorpusManager
from src.evaluation.comet import CometEvaluator
from src.models.qwen.inference import translate_with_qwen
from src.models.mistral.inference import translate_with_mistral

class _EvalHelper(BaseExperiment):
    experiment_id = 'notebook09'
    def build_training_tasks(self): raise NotImplementedError

helper = _EvalHelper(MasterCorpusManager())
test_pairs = helper.build_test_pairs('English', 'Ekegusii')
sources, references = test_pairs['source'].tolist(), test_pairs['target'].tolist()

translate_fn = translate_with_qwen if MODEL_NAME == 'qwen' else translate_with_mistral
predictions = translate_fn(sources, 'English', 'Ekegusii', adapter_path=ADAPTER_PATH)

comet_result = CometEvaluator().compute(predictions, references, sources)
print(f"Mean COMET Score: {comet_result['mean_score']:.4f}")
